# Hybrid Models for Hyperspectral Image Classification
**Autor:** Valerio Massimo Carioti (Student ID 1983063)

## Implementation Details

### Datasets
- **Datasets:** Indian Pines, Pavia University.
- **Pre-processing:** channel-wise normalization.
- **Splits:** 10 samples for the training set, 5 for the validation set, x for testing.

### Model
- **CTA-Net:**

### Experimental Setup
- **Hyperparameters:** ...

## Reproducibility Instructions
- **Requirements:** Python 3.12+, kagglehub, lightning, matplotlib, numpy, scikit-learn, scipy, torch, torchmetrics, torchvision, wandb.

## Code

In [ ]:
# Install repository
!git clone git@github.com:JoJohnny0/nn-project.git
!cd nn-project

In [ ]:
# Install dependencies
!pip install -r requirements.txt

In [ ]:
from typing import Literal

import kagglehub
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.io import loadmat
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
from torch.utils.data import DataLoader
from torchmetrics.functional import accuracy, cohen_kappa
import wandb

from modules.cta_net.cta_net import CTA_Lightning
from modules.dataset import get_loaders

In [ ]:
# Configuration
dataset: Literal['PaviaUniversity', 'IndianPines'] = 'PaviaUniversity'
seed: int|None = None

In [ ]:
# Data Hyperparameters
patch_size: int = 15
train_samples_per_class: int = 10
val_samples_per_class: int = 5
sigma: float = 0.1
central_region_size: int = 3

# Model Hyperparameters
hidden_channels: int = 128
heads: int = 2

# Training Hyperparameters
dropout: float = 0.1
lr: float = 8e-5
batch_size: int = 32
epochs: int = 150

In [ ]:
# Set random seed
if seed is not None:
    pl.seed_everything(seed)

### Data Handlig

In [ ]:
# Download dataset if needed
image: NDArray[np.uint16]
labels: NDArray[np.uint8]
if dataset == 'PaviaUniversity':
    dataset_path: str = kagglehub.dataset_download('syamkakarla/pavia-university-hsi')
    image = loadmat(f'{dataset_path}/PaviaU.mat')['paviaU']
    labels = loadmat(f'{dataset_path}/PaviaU_gt.mat')['paviaU_gt']
else:
    dataset_path: str = kagglehub.dataset_download('abhijeetgo/indian-pines-hyperspectral-dataset')
    image  = np.load(f'{dataset_path}/indianpinearray.npy')
    labels = np.load(f'{dataset_path}/IPgt.npy')

In [ ]:
# Get data loaders
train_loader: DataLoader[list[torch.Tensor]]
val_loader: DataLoader[list[torch.Tensor]]
test_loader: DataLoader[list[torch.Tensor]]
train_loader, val_loader, test_loader = get_loaders(image,
                                                    labels,
                                                    patch_size,
                                                    train_samples_per_class = train_samples_per_class,
                                                    val_samples_per_class = val_samples_per_class,
                                                    sigma = sigma,
                                                    central_region_size = central_region_size,
                                                    batch_size = batch_size
                                                    )

### Training

In [ ]:
# Initialize logger
wandb_logger: WandbLogger = WandbLogger(project = f"Hybrid Models for Hyperspectral Image Classification",
                                        name = f"CTA-Net_{dataset}_seed={seed}",
                                        save_dir = 'data/wandb',
                                        # Parameters not logged by the trainer
                                        config = {'train_samples_per_class': train_samples_per_class,
                                                  'val_samples_per_class': val_samples_per_class,
                                                  'sigma': sigma,
                                                  'central_region_size': central_region_size,
                                                  'batch_size': batch_size
                                                  }
                                        )
wandb_logger.experiment.define_metric('*', step_metric = 'epoch')

In [ ]:
# Add best checkpoint callback
save_best: ModelCheckpoint = ModelCheckpoint(monitor = 'val_loss',
                                             dirpath = 'data/checkpoints',
                                             filename = f'cta-net-epoch={{epoch}}-seed={seed}',
                                             save_weights_only = True
                                             )

In [ ]:
# Initialize model and trainer
n_classes: int = int(labels.max())
model: CTA_Lightning = CTA_Lightning(in_channels = image.shape[2],
                                        hidden_channels = hidden_channels,
                                        out_channels = n_classes,
                                        heads = heads,
                                        window_size = patch_size,
                                        dropout = dropout,
                                        lr = lr
                                        )
trainer: pl.Trainer = pl.Trainer(max_epochs = epochs,
                                    logger = wandb_logger,
                                    callbacks = save_best,
                                    log_every_n_steps = len(train_loader)
                                    )

In [ ]:
# Train the model
trainer.fit(model, train_loader, val_loader)

### Test

In [ ]:
# Get predictions
preds: torch.Tensor = torch.concat(trainer.predict(model, test_loader, ckpt_path = 'best'))  # type: ignore

# Get targets
targets: torch.Tensor = torch.concat([batch[1] for batch in test_loader])

In [ ]:
# General metrics
metrics: dict[str, float] = {'Overall Accuracy': accuracy(preds, targets, task = 'multiclass', average = 'micro', num_classes = n_classes).item(),
                             'Average Accuracy': accuracy(preds, targets, task = 'multiclass', average = 'macro', num_classes = n_classes).item(),
                             'Kappa': cohen_kappa(preds, targets, task = 'multiclass', num_classes = n_classes).item()
                             }

# Class-wise accuracy
conf_matrix: NDArray[np.int64] = confusion_matrix(targets, preds)
for i, class_acc in enumerate(conf_matrix.diagonal() / conf_matrix.sum(axis = 1)):
    metrics[f'Class {i + 1} Accuracy'] = class_acc.item()

In [ ]:
# Print metrics
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Display confusion matrix
disp: ConfusionMatrixDisplay = ConfusionMatrixDisplay(confusion_matrix = conf_matrix)
disp.plot()
plt.tight_layout()

In [ ]:
# Log to wandb
wandb_logger.experiment.log(metrics)
wandb_logger.experiment.log({'Confusion Matrix': wandb.Image(plt.gcf())})
wandb_logger.experiment.finish()
plt.close()